
### End-to-End Simulation Pipeline
This notebook demonstrates the generation of synthetic EIGER2 diffraction patterns 
designed for sub-pixel 6D geometric calibration. It replaces generative AI noise 
with strictly physics-bound, dynamically generated detector artifacts (via SVD).

In [ ]:
import os
import copy
import numpy as np
import torch
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter
import fabio

from pyFAI.calibrant import CALIBRANT_FACTORY
from pyFAI.detectors import Detector
from pyFAI.integrator.azimuthal import AzimuthalIntegrator

In [ ]:
dark_folder = r"C:\Users\gmurra12\projects\MAXIMA-ViT\datasets\CHESS 2011-11\darks_scan001"
dark_files = sorted([f for f in os.listdir(dark_folder) if f.endswith('.tiff')])

dark_images = []
for file in dark_files:
    img = fabio.open(os.path.join(dark_folder, file)).data
    dark_images.append(img)

N_darks = len(dark_images)
H, W = dark_images[0].shape
dark_stack = np.stack(dark_images, axis=0)

In [ ]:
flat_darks = dark_stack.reshape(N_darks, -1)
mean_dark_flat = np.mean(flat_darks, axis=0)
centered_darks = flat_darks - mean_dark_flat

print("Running SVD...")
U, S, Vt = np.linalg.svd(centered_darks, full_matrices=False)


true_weights_std = S / np.sqrt(N_darks)
k = N_darks - 2 
mean_dark = mean_dark_flat.reshape(H, W)
components = Vt[:k].reshape(k, H, W)
weights_std = true_weights_std[:k]

np.savez_compressed("dynamic_dark_priors.npz", 
                    mean_dark=mean_dark, 
                    components=components, 
                    weights_std=weights_std)

print(f"Saved top {k} physical dark components. Max variance: {weights_std[0]:.2f} counts.")

In [ ]:
class DynamicDarkGenerator:
    def __init__(self, priors_path: str):
        priors = np.load(priors_path)
        self.mean_dark = priors['mean_dark'].astype(np.float32)
        self.components = priors['components'].astype(np.float32)
        self.weights_std = priors['weights_std'].astype(np.float32)
        self.rng = np.random.default_rng()

    def generate(self) -> np.ndarray:
        weights = self.rng.normal(loc=0.0, scale=self.weights_std).astype(np.float32)
        
        dynamic_variation = np.tensordot(weights, self.components, axes=([0], [0]))
        synthetic_dark = self.mean_dark + dynamic_variation

        np.clip(synthetic_dark, a_min=0.0, a_max=None, out=synthetic_dark)
        return synthetic_dark

In [ ]:
class PhysicsDomainRandomizer:
    def __init__(self, image_shape: tuple[int, int], dark_priors_path: str):
        self.image_shape = image_shape
        self.rng = np.random.default_rng()
        self.dark_generator = DynamicDarkGenerator(dark_priors_path)

    def apply(self, clean_image: np.ndarray) -> np.ndarray:
        work = clean_image.astype(np.float32, copy=True)
        
        flux = self.rng.uniform(0.5, 5.0)
        work = np.clip(work * flux, 0.0, None)
        
        synthetic_dark = self.dark_generator.generate()
        work += synthetic_dark
        
        work = self.rng.poisson(work).astype(np.float32)
        
        return work

In [ ]:
def image_to_metrology_tensor(image: np.ndarray, target_size: int = 2048, crop_mode: str = "top_left"):
    """Converts image to tensor with physics-safe normalization and targeted cropping."""
    
    image = np.log1p(image)

    img_min, img_max = image.min(), image.max()
    if img_max > img_min:
        image = (image - img_min) / (img_max - img_min)
    else:
        image = np.zeros_like(image)

    image_tensor = torch.from_numpy(image).unsqueeze(0).repeat(3, 1, 1).float()
    
    orig_h, orig_w = image.shape
    pad_h = max(0, target_size - orig_h)
    pad_w = max(0, target_size - orig_w)
    
    if pad_h > 0 or pad_w > 0:
        image_tensor = F.pad(image_tensor, [pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2])
        
    if crop_mode == "top_left":
        final_tensor = F.crop(image_tensor, top=0, left=0, height=target_size, width=target_size)
        delta_y_pixels = 0.0  
        delta_x_pixels = 0.0
    else:
        final_tensor = F.center_crop(image_tensor, output_size=[target_size, target_size])
        delta_y_pixels = (target_size - orig_h) / 2.0
        delta_x_pixels = (target_size - orig_w) / 2.0

    final_tensor = F.normalize(final_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    return final_tensor, delta_y_pixels, delta_x_pixels

In [ ]:
def adjust_poni_labels(labels: dict, delta_y_pixels: float, delta_x_pixels: float, pixel_size_m: float = 75e-6):
    """Adjusts the Ground Truth PONI labels based on the tensor padding/cropping."""
    new_labels = copy.deepcopy(labels)
    new_labels['poni1'] += (delta_y_pixels * pixel_size_m)
    new_labels['poni2'] += (delta_x_pixels * pixel_size_m)
    return new_labels

In [ ]:
wavelength = 0.826e-10  # 15 keV
calibrant = CALIBRANT_FACTORY("alpha_Al2O3")
calibrant.wavelength = wavelength
detector = Detector(pixel1=200e-6, pixel2=200e-6, max_shape=(2048, 2048))

ground_truth = {
    "dist": 0.36, "poni1": 0.11, "poni2": 0.21,
    "rot1": 0.01, "rot2": -0.26, "rot3": 0.0
}

print("Generating clean pyFAI analytical rings...")
ai = AzimuthalIntegrator(detector=detector, wavelength=wavelength, **ground_truth)
clean_image = calibrant.fake_calibration_image(ai, Imax=16500, Imin=1917, resolution=0.15)

print("Injecting physical darks and Poisson noise...")
randomizer = PhysicsDomainRandomizer(detector.shape, "dynamic_dark_priors.npz")
noisy_image = randomizer.apply(clean_image)

print("Converting to tensor and shifting PONI labels...")
target_size = 2048
model_tensor, dy, dx = image_to_metrology_tensor(noisy_image, target_size=target_size)
adjusted_labels = adjust_poni_labels(ground_truth, dy, dx, pixel_size_m=detector.pixel1)

print("Loading real pattern...")
path = r""
pattern = fabio.open(path)
real_image = pattern.data


clean_log = np.log1p(clean_image)
noisy_log = np.log1p(noisy_image)
real_log = np.log1p(real_image)
view_tensor_np = model_tensor[0].numpy()


all_images_log = [clean_log, noisy_log, real_log]
global_vmin = min(img.min() for img in all_images_log)
global_vmax = max(img.max() for img in all_images_log)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))


axes[0].imshow(clean_log, cmap='viridis', vmin=global_vmin, vmax=global_vmax)
axes[0].set_title("1. Analytical pyFAI (Clean)")


axes[1].imshow(noisy_log, cmap='viridis', vmin=global_vmin, vmax=global_vmax)
axes[1].set_title("2. SVD Dark + Poisson Injection")


axes[2].imshow(real_log, cmap='viridis', vmin=global_vmin, vmax=global_vmax)
axes[2].set_title("3. Real Experimental Pattern")


axes[3].imshow(view_tensor_np, cmap='viridis')
axes[3].set_title(f"4. Final Tensor ({target_size}x{target_size})")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n--- Metrology Label Adjustment ---")
print(f"Y Pixel Shift: {dy} | X Pixel Shift: {dx}")
print(f"Original PONI1: {ground_truth['poni1']} m -> Adjusted: {adjusted_labels['poni1']} m")
print(f"Original PONI2: {ground_truth['poni2']} m -> Adjusted: {adjusted_labels['poni2']} m")

In [ ]:
# plot histogram of the clean, noisy, real, and final images
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(clean_image.flatten(), bins=100, alpha=0.5, label='Clean (Analytical)', color='blue', log=True)
ax.hist(noisy_image.flatten(), bins=100, alpha=0.5, label='Noisy (SVD Dark + Poisson)', color='orange', log=True)
ax.hist(real_image.flatten(), bins=100, alpha=0.5, label='Real Experimental', color='green', log=True)
ax.hist(view_tensor_np.flatten(), bins=100, alpha=0.5, label='Final (SVD Dark + Poisson)', color='red', log=True)
ax.set_title("Pixel Intensity Distribution (Log Scale)")
ax.set_xlabel("Pixel Intensity")
ax.set_ylabel("Frequency")
ax.legend()
plt.show()


In [ ]:
# project the final tensor to match the histogram of the real pattern for better visual comparison
from sklearn.preprocessing import QuantileTransformer
final_tensor_np = model_tensor[0].numpy().flatten().reshape(-1, 1)
real_flat = real_image.flatten().reshape(-1, 1)
quantile_transformer = QuantileTransformer(output_distribution='normal')
final_tensor_projected = quantile_transformer.fit_transform(final_tensor_np)
final_tensor_projected = final_tensor_projected.reshape(model_tensor[0].shape)
log_final_tensor_projected = np.log1p(final_tensor_projected)

plt.figure(figsize=(8, 8))
plt.imshow(log_final_tensor_projected, cmap='viridis', vmin=global_vmin, vmax=global_vmax)
plt.title("Final Tensor Projected to Real Histogram")
plt.axis('off')
plt.show()